# 🫁 Respiratory Sound Analysis

> Analysis of respiratory sounds using advanced signal processing techniques

---

## 📦 Installation & Setup

In [ ]:
!pip install -q kaggle numpy scipy matplotlib pandas librosa ordpy

In [ ]:
import sys
import shutil
import subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile

from google.colab import drive

MAX_FILES: int | None = None
EMBEDDING_DIM: int = 3
TIME_DELAY: int = 1

EXTRACTOR_CONFIG = {
    'n_fft': 2048,
    'hop_length': 512,
    'frame_length': 2048,
    'n_mfcc': 13,
    'epsilon': 1e-6,
}

In [ ]:
drive.mount('/content/drive')


## ⚙️ Configure Kaggle & Download Datasets Every Run

In [ ]:
KAGGLE_DIR = Path.home() / '.kaggle'
KAGGLE_JSON = KAGGLE_DIR / 'kaggle.json'
KAGGLE_DRIVE = Path('/content/drive/MyDrive/kaggle.json')

KAGGLE_DIR.mkdir(exist_ok=True)
shutil.copy(KAGGLE_DRIVE, KAGGLE_JSON)
KAGGLE_JSON.chmod(0o600)

KAGGLE_DATASETS = {
    'respiratory_sound_database': 'vbookshelf/respiratory-sound-database',
    'lung_dataset': 'arashnic/lung-dataset',
    'asthma_detection_v2': 'mohammedtawfikmusaed/asthma-detection-dataset-version-2',
    'respiratory_processed_audio': 'shivam316/respiratory-disease-dataset-processed-audio-files',
}

DATA_ROOT = Path('/content/datasets')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def download_kaggle_dataset(dataset_slug: str, target_dir: Path) -> None:
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        'kaggle', 'datasets', 'download',
        '-d', dataset_slug,
        '--path', str(target_dir),
        '--unzip',
        '-q',
    ]
    subprocess.run(cmd, check=True)

In [ ]:
for dataset_name, dataset_slug in KAGGLE_DATASETS.items():
    target = DATA_ROOT / dataset_name
    print(f'⬇️ Downloading {dataset_slug} -> {target}')
    download_kaggle_dataset(dataset_slug, target)
    print(f'✅ Ready: {dataset_name}')

DATASET_PATH = DATA_ROOT / 'respiratory_sound_database'
print(f'\n📁 Primary dataset path for this notebook: {DATASET_PATH}')

## 📥 Clone Analysis Repository

In [ ]:
REPO_PATH = Path('/content/course_paper')

!git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
!git -C {REPO_PATH} sparse-checkout set app

sys.path.insert(0, str(REPO_PATH))

from app.features import (
    AudioFeatureExtractor,
    compute_entropy_complexity,
    extract_frequency_features,
 )

## 🔊 Extract Features and Create Unified DataFrame

This notebook now extracts from all downloaded datasets and combines everything in one table.

Compact clinical golden set (14 features):
- Base: low_freq_energy, spectral_centroid, entropy, complexity
- Added: zcr_variance, mfcc_1..3 mean/var, spectral_flatness, hjorth_mobility, rmse_coeff_var

In [ ]:
signals = {}
dataset_file_counts = {}

for dataset_name in KAGGLE_DATASETS:
    dataset_root = DATA_ROOT / dataset_name
    all_wav_files = sorted(dataset_root.rglob('*.wav'))
    selected_files = all_wav_files if MAX_FILES is None else all_wav_files[:MAX_FILES]
    dataset_file_counts[dataset_name] = len(selected_files)

    for wav_path in selected_files:
        try:
            sample_rate, signal_data = wavfile.read(str(wav_path))
        except Exception as exc:
            print(f'⚠️ Skipped {wav_path} ({exc})')
            continue

        signal_data = signal_data.astype(np.float64)
        signal_data = signal_data[:, 0] if signal_data.ndim > 1 else signal_data

        relative_path = wav_path.relative_to(dataset_root).as_posix()
        file_id = f'{dataset_name}::{relative_path}'

        signals[file_id] = {
            'signal': signal_data,
            'sample_rate': int(sample_rate),
            'dataset_name': dataset_name,
            'wav_path': wav_path,
            'relative_path': relative_path,
        }

print('✅ Loaded files by dataset:')
for dataset_name, count in dataset_file_counts.items():
    print(f' - {dataset_name}: {count}')
print(f'\n✅ Total loaded files: {len(signals)}')

In [ ]:
resp_root = DATA_ROOT / 'respiratory_sound_database'
resp_diagnosis_files = list(resp_root.rglob('patient_diagnosis.csv'))
resp_diagnosis_map = {}

if resp_diagnosis_files:
    diagnosis_df = pd.read_csv(resp_diagnosis_files[0])
    patient_col, diagnosis_col = diagnosis_df.columns[0], diagnosis_df.columns[1]
    resp_diagnosis_map = dict(zip(diagnosis_df[patient_col], diagnosis_df[diagnosis_col]))
    print(f"📋 Respiratory diagnosis map loaded from: {resp_diagnosis_files[0]}")
else:
    print('⚠️ patient_diagnosis.csv not found for respiratory_sound_database')

def parse_patient_id(filename: str) -> int:
    stem = Path(filename).stem
    prefix = stem.split('_')[0]
    return int(prefix) if prefix.isdigit() else -1

# Give each non-numeric patient prefix its own normal numeric id.
known_patient_ids = []
unknown_patient_prefixes = []

for meta in signals.values():
    parsed_id = parse_patient_id(meta['wav_path'].name)
    if parsed_id != -1:
        known_patient_ids.append(parsed_id)
    else:
        unknown_patient_prefixes.append(Path(meta['wav_path'].name).stem.split('_')[0])

next_generated_id = (max(known_patient_ids) + 1) if known_patient_ids else 1
unknown_patient_id_map = {
    prefix: idx
    for idx, prefix in enumerate(sorted(set(unknown_patient_prefixes)), start=next_generated_id)
}

def resolve_patient_id(filename: str) -> int:
    parsed_id = parse_patient_id(filename)
    if parsed_id != -1:
        return parsed_id

    prefix = Path(filename).stem.split('_')[0]
    return unknown_patient_id_map[prefix]

def infer_diagnosis(meta: dict, patient_id: int) -> str:
    dataset_name = meta['dataset_name']

    if dataset_name == 'respiratory_sound_database' and patient_id in resp_diagnosis_map:
        return str(resp_diagnosis_map[patient_id])

    parent_name = meta['wav_path'].parent.name.strip()
    generic_dirs = {'audio', 'audio_and_txt_files', 'wav', 'wavs', 'files'}

    if parent_name and parent_name.lower() not in generic_dirs and not parent_name.isdigit():
        return parent_name

    return 'Unknown'

extractor = AudioFeatureExtractor(**EXTRACTOR_CONFIG)
rows = []

for file_id, meta in signals.items():
    signal_data = meta['signal']
    sample_rate = meta['sample_rate']
    patient_id = resolve_patient_id(meta['wav_path'].name)

    _, _, freq_features = extract_frequency_features(signal_data, sample_rate)
    entropy, complexity = compute_entropy_complexity(
        signal_data,
        embedding_dim=EMBEDDING_DIM,
        time_delay=TIME_DELAY,
    )
    advanced_features = extractor.extract_golden(signal_data, sample_rate)

    total_energy = (
        freq_features['low_freq_energy']
        + freq_features['mid_freq_energy']
        + freq_features['high_freq_energy']
    ) + 1e-10

    diagnosis = infer_diagnosis(meta, patient_id)
    meta['diagnosis'] = diagnosis

    row = {
        'file_id': file_id,
        'filename': meta['wav_path'].name,
        'relative_path': meta['relative_path'],
        'dataset': meta['dataset_name'],
        'patient_id': patient_id,
        'low_freq_energy': freq_features['low_freq_energy'] / total_energy,
        'spectral_centroid': freq_features['spectral_centroid'],
        'entropy': entropy,
        'complexity': complexity,
        'diagnosis': diagnosis,
        **advanced_features,
    }
    rows.append(row)

results_df = pd.DataFrame(rows)
if results_df.empty:
    raise ValueError('No rows extracted. Check dataset download and WAV file discovery.')

feature_only_cols = [
    c for c in results_df.columns
    if c not in {'file_id', 'filename', 'relative_path', 'dataset', 'patient_id', 'diagnosis'}
]

print(f"\n✅ Extracted shape: {results_df.shape}")
print(f"✅ Feature count: {len(feature_only_cols)}")
print(f"\n📋 By dataset:\n{results_df['dataset'].value_counts()}")
print(f"\n📋 By diagnosis:\n{results_df['diagnosis'].value_counts().head(20)}")
display(results_df.head())

In [ ]:
# Golden feature set used for aggregation and plotting
feature_cols = [
    'low_freq_energy',
    'spectral_centroid',
    'entropy',
    'complexity',
    'zcr_variance',
    'mfcc_1_mean',
    'mfcc_2_mean',
    'mfcc_3_mean',
    'mfcc_1_var',
    'mfcc_2_var',
    'mfcc_3_var',
    'spectral_flatness',
    'hjorth_mobility',
    'rmse_coeff_var',
]
feature_cols = [col for col in feature_cols if col in results_df.columns]

avg_by_diagnosis = results_df.groupby('diagnosis')[feature_cols].mean().reset_index()
display(avg_by_diagnosis)

In [ ]:
from google.colab import files

filename = 'combined_respiratory_features.csv'
results_df.to_csv(filename, index=False)
files.download(filename)
print(f'⬇️ Downloaded: {filename} ({len(results_df)} rows)')

## 📊 Visualizations

In [ ]:
DIAGNOSIS_COLORS = [
    '#FF4757', '#1DD1A1', '#5F9EFF', '#FFA502', '#00D2C3', 
    '#FFE66D', '#C56CF0', '#54A0FF', '#FF6B9D', '#48DBFB', 
    '#F8EFBA', '#1E90FF', '#FF7979'
]
PLOT_STYLE = {
    'grid_alpha': 0.15,
    'bar_alpha': 0.85,
    'line_width': 2.0,
    'title_fontsize': 15,
    'label_fontsize': 12,
    'value_fontsize': 9,
    'marker_size': 4,
    'marker_density': 20,
    'ylim_multiplier': 1.20,
    'scatter_size': 300
}

plt.style.use('dark_background')

plt.rcParams['figure.facecolor'] = '#0B1929'
plt.rcParams['axes.facecolor'] = '#1A2332'
plt.rcParams['savefig.facecolor'] = '#0B1929'


In [ ]:
def save_plot(filename: str, dpi: int = 600):
    from google.colab import files
    local_path = f'/content/{filename}'
    plt.savefig(local_path, dpi=dpi, bbox_inches='tight')
    files.download(local_path)
    print(f'⬇️ Downloaded: {filename}')

In [ ]:
num_diagnoses = len(avg_by_diagnosis)
n_features = len(feature_cols)
n_cols = 4
n_rows = int(np.ceil(n_features / n_cols))

fig_width = max(16, num_diagnoses * 1.5)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for idx, feature in enumerate(feature_cols):
    ax = axes[idx]

    diagnoses = avg_by_diagnosis['diagnosis']
    values = avg_by_diagnosis[feature]

    colors = [DIAGNOSIS_COLORS[i % len(DIAGNOSIS_COLORS)] for i in range(num_diagnoses)]
    bars = ax.bar(
        range(num_diagnoses),
        values,
        color=colors,
        alpha=PLOT_STYLE['bar_alpha'],
        width=0.7 if num_diagnoses <= 8 else 0.6,
    )

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height(),
            f'{val:.3f}',
            ha='center',
            va='bottom',
            fontsize=PLOT_STYLE['value_fontsize'] if num_diagnoses <= 8 else 8,
            fontweight='600',
        )

    ax.set_title(feature.replace('_', ' ').title(), fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks(range(num_diagnoses))
    ax.set_xticklabels(
        diagnoses,
        rotation=45,
        ha='right',
        fontsize=PLOT_STYLE['value_fontsize'] if num_diagnoses <= 8 else 7,
    )
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], axis='y', zorder=0)
    ax.set_ylim(0, values.max() * PLOT_STYLE['ylim_multiplier'])
    ax.set_axisbelow(True)

for idx in range(n_features, len(axes)):
    axes[idx].axis('off')

plt.suptitle(
    'Feature Comparison Across Diagnoses',
    fontsize=PLOT_STYLE['title_fontsize'] + 2,
    fontweight='bold',
    y=0.995,
 )
plt.tight_layout()
save_plot('feature_comparison.png', dpi=600)
plt.show()

In [ ]:
signals_by_diagnosis = defaultdict(list)
sample_rates_by_diagnosis = defaultdict(list)

for meta in signals.values():
    diagnosis = meta.get('diagnosis', 'Unknown')
    signals_by_diagnosis[diagnosis].append(meta['signal'])
    sample_rates_by_diagnosis[diagnosis].append(meta['sample_rate'])

for idx, diagnosis in enumerate(avg_by_diagnosis['diagnosis']):
    diag_signals = signals_by_diagnosis.get(diagnosis, [])
    diag_rates = sample_rates_by_diagnosis.get(diagnosis, [])

    if not diag_signals:
        continue

    sr_counts = pd.Series(diag_rates).value_counts()
    target_sr = int(sr_counts.index[0])

    aligned_signals = [
        signal for signal, sr in zip(diag_signals, diag_rates)
        if sr == target_sr
    ]

    if not aligned_signals:
        continue

    min_length = min(len(signal) for signal in aligned_signals)
    signals_array = np.array([signal[:min_length] for signal in aligned_signals], dtype=np.float64)

    mean_signal = np.mean(signals_array, axis=0)
    std_signal = np.std(signals_array, axis=0)
    time_seconds = np.arange(len(mean_signal)) / target_sr
    file_count = len(aligned_signals)
    color = DIAGNOSIS_COLORS[idx % len(DIAGNOSIS_COLORS)]

    fig, ax = plt.subplots(figsize=(12, 4.5))

    ax.fill_between(
        time_seconds,
        mean_signal - std_signal,
        mean_signal + std_signal,
        color=color,
        alpha=0.3,
        label='±1 SD',
        zorder=2,
    )

    ax.plot(
        time_seconds,
        mean_signal,
        color=color,
        linewidth=PLOT_STYLE['line_width'] + 0.5,
        label='Mean',
        alpha=0.95,
        zorder=3,
    )

    marker_interval = max(1, len(time_seconds) // PLOT_STYLE['marker_density'])
    ax.plot(
        time_seconds[::marker_interval],
        mean_signal[::marker_interval],
        'o',
        color=color,
        markersize=PLOT_STYLE['marker_size'],
        zorder=4,
    )

    duration = len(mean_signal) / target_sr
    ax.set_title(
        f'{diagnosis} - (observations = {file_count}, duration = {duration:.2f}s, sr = {target_sr})',
        fontsize=PLOT_STYLE['title_fontsize'],
        fontweight='bold',
        pad=15,
    )
    ax.set_xlabel('Time (seconds)', fontsize=PLOT_STYLE['label_fontsize'], fontweight='500')
    ax.set_ylabel('Amplitude', fontsize=PLOT_STYLE['label_fontsize'], fontweight='500')
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], zorder=0)
    ax.set_axisbelow(True)

    legend = ax.legend(loc='upper right', fontsize=10, framealpha=0.9, fancybox=True, shadow=False)
    legend.get_frame().set_linewidth(1.0)

    plt.tight_layout()
    safe_name = ''.join(ch if ch.isalnum() or ch in {'_', '-'} else '_' for ch in str(diagnosis))
    save_plot(f'{safe_name}_signal.png', dpi=600)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

for idx, row in avg_by_diagnosis.iterrows():
    diagnosis = row['diagnosis']
    entropy = row['entropy']
    complexity = row['complexity']
    color = DIAGNOSIS_COLORS[idx % len(DIAGNOSIS_COLORS)]
    
    ax.scatter(entropy, complexity, s=PLOT_STYLE['scatter_size'], color=color, 
              alpha=PLOT_STYLE['bar_alpha'], label=diagnosis, zorder=3)
    
    ax.annotate(diagnosis, (entropy, complexity), 
               xytext=(10, 10), textcoords='offset points',
               fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3),
               arrowprops=dict(arrowstyle='-', lw=0.8, alpha=0.6))

ax.set_xlabel('Entropy', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('Complexity', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Entropy X Complexity', fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], linestyle='--', zorder=0)
ax.set_axisbelow(True)

legend = ax.legend(loc='best', fontsize=10, framealpha=0.95, fancybox=True, shadow=False)
legend.get_frame().set_linewidth(1.2)

plt.tight_layout()
save_plot('entropy_complexity.png', dpi=600)
plt.show()
